# 01_sol: Progressive Banking Tool-Use Agent (Stage-Gated)

Timebox: **55 minutes**  
Language: **Python (Colab)**

This mock combines:
1. Tool-use agent loop implementation.
2. Prompt + schema shaping.
3. Progressive complexity in CodeSignal-style levels.

## Scenario
Build a support agent for a small banking workflow. The model can call local tools for balances, transfers, and audits.

## Stage-gated levels
- **Level 1:** Prompt + schema + single tool call loop.
- **Level 2:** Multiple tool calls in one turn with strict sequencing.
- **Level 3:** Business-logic mutation + safe error handling.
- **Level 4:** `pause_turn` continuation + bounded retry + max-steps safety.

## What to implement
1. `build_system_prompt`
2. `build_tool_schemas`
3. `execute_tool_call`
4. `run_agent`

## Time guidance
- 10 min: read staged tests and lock contracts.
- 35 min: implement minimal passing behavior for all levels.
- 10 min: harden edge cases and re-run stage sequence.


In [ ]:
# Chunk overview: prepare deterministic fixtures and scripted model behavior.

import inspect
import json
from copy import deepcopy
from typing import Any, Callable

ACCOUNTS = {
    "a-100": {"account_id": "a-100", "balance": 300.0},
    "a-200": {"account_id": "a-200", "balance": 120.0},
}

LEDGER = {
    "a-100": [
        {"kind": "deposit", "amount": 300.0},
    ],
    "a-200": [
        {"kind": "deposit", "amount": 120.0},
    ],
}

TRANSFERS: list[dict[str, Any]] = []
AUDIT_ATTEMPTS: dict[str, int] = {}


def reset_state() -> None:
    ACCOUNTS["a-100"]["balance"] = 300.0
    ACCOUNTS["a-200"]["balance"] = 120.0
    LEDGER["a-100"] = [{"kind": "deposit", "amount": 300.0}]
    LEDGER["a-200"] = [{"kind": "deposit", "amount": 120.0}]
    TRANSFERS.clear()
    AUDIT_ATTEMPTS.clear()


def get_balance(account_id: str) -> dict[str, Any]:
    if account_id not in ACCOUNTS:
        raise ValueError("unknown_account")
    return deepcopy(ACCOUNTS[account_id])


def list_recent_transactions(account_id: str, limit: int = 3) -> list[dict[str, Any]]:
    if account_id not in LEDGER:
        raise ValueError("unknown_account")
    return deepcopy(LEDGER[account_id][-limit:])


def transfer_funds(from_account: str, to_account: str, amount: float) -> dict[str, Any]:
    if from_account not in ACCOUNTS or to_account not in ACCOUNTS:
        raise ValueError("unknown_account")
    if amount <= 0:
        raise ValueError("amount_must_be_positive")
    if ACCOUNTS[from_account]["balance"] < amount:
        raise ValueError("insufficient_funds")

    ACCOUNTS[from_account]["balance"] -= amount
    ACCOUNTS[to_account]["balance"] += amount
    out = {"from": from_account, "to": to_account, "amount": amount, "status": "posted"}

    LEDGER[from_account].append({"kind": "transfer_out", "amount": amount, "to": to_account})
    LEDGER[to_account].append({"kind": "transfer_in", "amount": amount, "from": from_account})
    TRANSFERS.append(deepcopy(out))
    return out


def run_account_audit(account_id: str) -> dict[str, Any]:
    count = AUDIT_ATTEMPTS.get(account_id, 0)
    AUDIT_ATTEMPTS[account_id] = count + 1
    if count == 0:
        raise RuntimeError("transient_audit_backend_timeout")
    return {"account_id": account_id, "status": "audit_ok"}


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "get_balance": get_balance,
    "list_recent_transactions": list_recent_transactions,
    "transfer_funds": transfer_funds,
    "run_account_audit": run_account_audit,
}


class ScriptedModel:
    def __init__(self, responses: list[dict[str, Any]]) -> None:
        self._responses = deepcopy(responses)
        self._index = 0
        self.last_system_prompt: str | None = None
        self.last_tools: list[dict[str, Any]] = []

    def __call__(self, messages: list[dict[str, Any]], system_prompt: str, tools: list[dict[str, Any]]) -> dict[str, Any]:
        self.last_system_prompt = system_prompt
        self.last_tools = deepcopy(tools)
        if self._index >= len(self._responses):
            return {"stop_reason": "end_turn", "content": [{"type": "text", "text": "no scripted response"}]}
        response = self._responses[self._index]
        self._index += 1
        return deepcopy(response)


In [ ]:
# Chunk overview: implement a robust, stage-friendly reference solution with deterministic behavior.

def build_system_prompt() -> str:
    return (
        "You are a banking support assistant. "
        "Use tools before making account or transaction claims. "
        "Never fabricate tool outputs. "
        "If a transfer request is missing required fields, ask a short clarifying question. "
        "Keep responses concise and grounded in tool results."
    )


def build_tool_schemas(tool_registry: dict[str, Callable[..., Any]]) -> list[dict[str, Any]]:
    schemas: list[dict[str, Any]] = []
    for name, fn in sorted(tool_registry.items()):
        sig = inspect.signature(fn)
        props: dict[str, dict[str, str]] = {}
        required: list[str] = []
        for param in sig.parameters.values():
            props[param.name] = {"type": "string"}
            if param.default is inspect._empty:
                required.append(param.name)
        schemas.append(
            {
                "name": name,
                "description": f"Call {name} for deterministic backend data.",
                "input_schema": {
                    "type": "object",
                    "properties": props,
                    "required": required,
                    "additionalProperties": True,
                },
            }
        )
    return schemas


def _validate_tool_call(tool_call: dict[str, Any], tool_registry: dict[str, Callable[..., Any]]) -> str | None:
    required_fields = {"id", "name", "input"}
    if not required_fields.issubset(tool_call):
        return "tool_call_missing_required_fields"

    tool_name = tool_call["name"]
    payload = tool_call["input"]

    if tool_name not in tool_registry:
        return "unknown_tool"
    if not isinstance(payload, dict):
        return "tool_input_must_be_object"

    sig = inspect.signature(tool_registry[tool_name])
    missing = [
        p.name for p in sig.parameters.values() if p.default is inspect._empty and p.name not in payload
    ]
    if missing:
        return f"missing_required_args:{','.join(sorted(missing))}"

    return None


def execute_tool_call(
    tool_call: dict[str, Any],
    tool_registry: dict[str, Callable[..., Any]],
    cache: dict[str, dict[str, Any]],
) -> dict[str, Any]:
    tool_id = str(tool_call.get("id", "missing_id"))
    tool_name = str(tool_call.get("name", "missing_name"))

    err = _validate_tool_call(tool_call, tool_registry)
    if err:
        return {
            "type": "tool_result",
            "tool_use_id": tool_id,
            "is_error": True,
            "from_cache": False,
            "content": json.dumps({"error": err, "name": tool_name}, sort_keys=True),
        }

    payload = tool_call["input"]
    cache_key = f"{tool_name}:{json.dumps(payload, sort_keys=True)}"
    if cache_key in cache:
        cached = deepcopy(cache[cache_key])
        cached["tool_use_id"] = tool_id
        cached["from_cache"] = True
        return cached

    attempt = 0
    while True:
        try:
            result = tool_registry[tool_name](**payload)
            message = {
                "type": "tool_result",
                "tool_use_id": tool_id,
                "is_error": False,
                "from_cache": False,
                "content": json.dumps({"result": result}, sort_keys=True),
            }
            cache[cache_key] = deepcopy(message)
            return message
        except RuntimeError as exc:
            if attempt == 0:
                attempt += 1
                continue
            return {
                "type": "tool_result",
                "tool_use_id": tool_id,
                "is_error": True,
                "from_cache": False,
                "content": json.dumps({"error": str(exc)}, sort_keys=True),
            }
        except Exception as exc:
            return {
                "type": "tool_result",
                "tool_use_id": tool_id,
                "is_error": True,
                "from_cache": False,
                "content": json.dumps({"error": str(exc)}, sort_keys=True),
            }


def run_agent(
    user_prompt: str,
    model: Callable[[list[dict[str, Any]], str, list[dict[str, Any]]], dict[str, Any]],
    tool_registry: dict[str, Callable[..., Any]],
    max_steps: int = 8,
) -> dict[str, Any]:
    messages: list[dict[str, Any]] = [{"role": "user", "content": [{"type": "text", "text": user_prompt}]}]
    system_prompt = build_system_prompt()
    tools = build_tool_schemas(tool_registry)
    cache: dict[str, dict[str, Any]] = {}
    stats = {"tool_calls": 0, "cache_hits": 0}

    for _ in range(max_steps):
        response = model(messages, system_prompt, tools)
        stop_reason = response.get("stop_reason")
        content = response.get("content", [])
        if not isinstance(content, list):
            raise RuntimeError("assistant_content_must_be_list")

        messages.append({"role": "assistant", "content": content})

        if stop_reason == "tool_use":
            tool_calls = [b for b in content if b.get("type") == "tool_use"]
            if not tool_calls:
                raise RuntimeError("tool_use_without_blocks")
            tool_results: list[dict[str, Any]] = []
            for call in tool_calls:
                result = execute_tool_call(call, tool_registry, cache)
                stats["tool_calls"] += 1
                if result.get("from_cache"):
                    stats["cache_hits"] += 1
                tool_results.append(result)
            messages.append({"role": "user", "content": tool_results})
            continue

        if stop_reason == "pause_turn":
            continue

        if stop_reason == "end_turn":
            final_text = " ".join(
                block.get("text", "").strip() for block in content if block.get("type") == "text"
            ).strip()
            return {"final_text": final_text, "messages": messages, "stats": stats}

        raise RuntimeError(f"unsupported_stop_reason:{stop_reason}")

    raise RuntimeError("max_steps_exceeded")


In [ ]:
# Chunk overview: stage-gated checks that emulate progressive complexity.

def _tool_result_blocks(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
    blocks: list[dict[str, Any]] = []
    for m in messages:
        if m.get("role") != "user":
            continue
        for b in m.get("content", []):
            if b.get("type") == "tool_result":
                blocks.append(b)
    return blocks


def run_level_1() -> None:
    reset_state()
    prompt = build_system_prompt().lower()
    assert "use tools" in prompt
    assert "never fabricate" in prompt

    defs = build_tool_schemas(TOOL_REGISTRY)
    assert sorted(d["name"] for d in defs) == sorted(TOOL_REGISTRY.keys())

    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "content": [
                    {"type": "tool_use", "id": "l1-1", "name": "get_balance", "input": {"account_id": "a-100"}}
                ],
            },
            {"stop_reason": "end_turn", "content": [{"type": "text", "text": "Balance is 300."}]},
        ]
    )
    result = run_agent("check balance", model, TOOL_REGISTRY)
    assert "300" in result["final_text"]


def run_level_2() -> None:
    reset_state()
    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "content": [
                    {"type": "tool_use", "id": "l2-1", "name": "list_recent_transactions", "input": {"account_id": "a-100", "limit": 1}},
                    {"type": "tool_use", "id": "l2-2", "name": "get_balance", "input": {"account_id": "a-100"}},
                ],
            },
            {"stop_reason": "end_turn", "content": [{"type": "text", "text": "latest tx + balance ready"}]},
        ]
    )
    result = run_agent("details", model, TOOL_REGISTRY)
    blocks = _tool_result_blocks(result["messages"])
    assert len(blocks) == 2

    assistant_idx = next(i for i, m in enumerate(result["messages"]) if m["role"] == "assistant")
    assert result["messages"][assistant_idx + 1]["role"] == "user"
    assert all(b.get("type") == "tool_result" for b in result["messages"][assistant_idx + 1]["content"])


def run_level_3() -> None:
    reset_state()
    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "content": [
                    {"type": "tool_use", "id": "l3-1", "name": "transfer_funds", "input": {"from_account": "a-100", "to_account": "a-200", "amount": 25.0}},
                    {"type": "tool_use", "id": "l3-2", "name": "not_a_tool", "input": {"x": 1}},
                ],
            },
            {"stop_reason": "end_turn", "content": [{"type": "text", "text": "transfer attempted"}]},
        ]
    )
    result = run_agent("transfer", model, TOOL_REGISTRY)

    assert ACCOUNTS["a-100"]["balance"] == 275.0
    assert ACCOUNTS["a-200"]["balance"] == 145.0

    blocks = _tool_result_blocks(result["messages"])
    bad = [b for b in blocks if b["tool_use_id"] == "l3-2"][0]
    assert bad["is_error"] is True
    assert "unknown_tool" in bad["content"]


def run_level_4() -> None:
    reset_state()
    model = ScriptedModel(
        [
            {"stop_reason": "pause_turn", "content": [{"type": "text", "text": "processing"}]},
            {
                "stop_reason": "tool_use",
                "content": [
                    {"type": "tool_use", "id": "l4-1", "name": "run_account_audit", "input": {"account_id": "a-100"}}
                ],
            },
            {"stop_reason": "end_turn", "content": [{"type": "text", "text": "audit complete"}]},
        ]
    )
    result = run_agent("audit", model, TOOL_REGISTRY)
    assert "audit complete" in result["final_text"]
    assert AUDIT_ATTEMPTS["a-100"] == 2

    loop_model = ScriptedModel(
        [
            {"stop_reason": "tool_use", "content": [{"type": "tool_use", "id": "mx-1", "name": "get_balance", "input": {"account_id": "a-100"}}]},
            {"stop_reason": "tool_use", "content": [{"type": "tool_use", "id": "mx-2", "name": "get_balance", "input": {"account_id": "a-100"}}]},
        ]
    )
    try:
        run_agent("loop", loop_model, TOOL_REGISTRY, max_steps=1)
        raise AssertionError("Expected max_steps_exceeded")
    except RuntimeError as exc:
        assert "max_steps_exceeded" in str(exc)


def run_exam01_tests() -> None:
    run_level_1()
    print("Level 1 passed")
    run_level_2()
    print("Level 2 passed")
    run_level_3()
    print("Level 3 passed")
    run_level_4()
    print("Level 4 passed")
    print("01_mock tests passed")


run_exam01_tests()
